# Food Insecurity and Income During COVID-19

**Research question:** Did low-income households experience significantly higher food insecurity rates than high-income households during the COVID-19 pandemic (2020–2021)?

**Data:** U.S. Census Bureau Household Pulse Survey — weekly aggregated food sufficiency counts by income group, 2020–2021.

**Source:** https://www.kaggle.com/datasets/jackogozaly/pulse-survey-food-insecurity-data

---
## Step 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## Step 2: Load and Describe the Data

Each row in the dataset is an aggregated population estimate for one income group in one survey week at one location. The food columns contain counts (how many people reported each food sufficiency level).

In [ ]:
# Load the data
df = pd.read_csv('Final_Pulse_Data.csv', low_memory=False)
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Check data types and missing values
df.info()

In [ ]:
# Check unique values for key columns
print('Income groups:')
print(df['Income'].dropna().unique())
print('\nLocations (sample):', df['Location'].unique()[:10])
print('\nYears:', df['Year'].unique())
print('\nNumber of unique weeks:', df['Week'].nunique())

---
## Step 3: Clean and Filter the Data

We filter to:
- National-level rows only (`Location == 'US'`) to avoid double-counting state data
- Rows where `Income` is filled in (not NaN and not 'Did not report')

Then we compute a **food insecurity rate** for each row.

In [ ]:
# Filter to national-level income rows
inc = df[df['Income'].notna() & (df['Location'] == 'US')].copy()
inc = inc[inc['Income'] != 'Did not report']

print(f'Rows after filtering: {len(inc)}')
print(f'Income groups: {inc["Income"].nunique()}')
print(f'Survey weeks: {inc["Week"].nunique()}')

In [ ]:
# Compute food insecurity rate per row
# Total = sum of all food sufficiency categories
inc['total'] = (
    inc['Enough of the kinds of food wanted'] +
    inc['Enough Food, but not always the kinds wanted'] +
    inc['Sometimes not enough to eat'] +
    inc['Often not enough to eat'] +
    inc['Did not report']
)

# Insecure = sometimes + often not enough to eat
inc['insecure_count'] = (
    inc['Sometimes not enough to eat'] +
    inc['Often not enough to eat']
)

# Insecurity rate = proportion of total
inc['insecurity_rate'] = inc['insecure_count'] / inc['total']

inc[['Income', 'Week', 'Year', 'total', 'insecure_count', 'insecurity_rate']].head(10)

---
## Step 4: Descriptive Statistics

Before running any tests, we explore the data to understand the patterns.

In [ ]:
# Summary statistics by income group
income_order = [
    'Less than $25,000', '$25,000 - $34,999', '$35,000 - $49,999',
    '$50,000 - $74,999', '$75,000 - $99,999', '$100,000 - $149,999',
    '$150,000 - $199,999', '$200,000 and above'
]

summary = inc.groupby('Income')['insecurity_rate'].agg(['mean', 'median', 'std', 'count'])
summary.columns = ['Mean Rate', 'Median Rate', 'Std Dev', 'N Weeks']
summary = summary.loc[income_order]
print(summary.round(4))

In [ ]:
# Bar chart: mean insecurity rate by income group
plt.figure(figsize=(10, 5))
plt.bar(range(len(income_order)), summary['Mean Rate'],
        color='steelblue', edgecolor='black')
plt.xticks(range(len(income_order)), income_order, rotation=30, ha='right')
plt.title('Mean Food Insecurity Rate by Income Group\n(U.S. Household Pulse Survey, 2020–2021)')
plt.ylabel('Mean Food Insecurity Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Define low-income and high-income groups
low_inc_groups = ['Less than $25,000', '$25,000 - $34,999']
high_inc_groups = ['$75,000 - $99,999', '$100,000 - $149,999',
                   '$150,000 - $199,999', '$200,000 and above']

# Line chart: insecurity rate over time for low vs high income
low_by_week = inc[inc['Income'].isin(low_inc_groups)].groupby('Week')['insecurity_rate'].mean()
high_by_week = inc[inc['Income'].isin(high_inc_groups)].groupby('Week')['insecurity_rate'].mean()

plt.figure(figsize=(12, 5))
plt.plot(range(len(low_by_week)), low_by_week.values,
         'o-', label='Low Income (< $35k)', color='tomato')
plt.plot(range(len(high_by_week)), high_by_week.values,
         's-', label='High Income ($75k+)', color='steelblue')
plt.title('Food Insecurity Rate Over Time by Income Group\n(2020–2021)')
plt.xlabel('Survey Week (in order)')
plt.ylabel('Mean Food Insecurity Rate')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot comparing low vs high income distributions
low_rates = inc[inc['Income'].isin(low_inc_groups)]['insecurity_rate'].values
high_rates = inc[inc['Income'].isin(high_inc_groups)]['insecurity_rate'].values

plt.figure(figsize=(7, 5))
plt.boxplot([low_rates, high_rates],
            labels=['Low Income\n(< $35k)', 'High Income\n($75k+)'],
            patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))
plt.title('Distribution of Weekly Food Insecurity Rates\nLow vs. High Income Groups')
plt.ylabel('Food Insecurity Rate')
plt.tight_layout()
plt.show()

print(f'Low-income mean:  {low_rates.mean():.4f}')
print(f'High-income mean: {high_rates.mean():.4f}')
print(f'Observed difference (low - high): {low_rates.mean() - high_rates.mean():.4f}')

---
## Step 5: Permutation Test

**Null hypothesis:** There is no difference in food insecurity rates between low-income and high-income households. Any observed difference is due to chance — as if the income labels were randomly assigned.

**Alternative hypothesis:** Low-income households have higher food insecurity rates than high-income households.

**Test statistic:** Mean insecurity rate (low-income) − mean insecurity rate (high-income)

**Method:** Pool all weekly rates from both groups, randomly shuffle the group labels 10,000 times, recompute the difference each time.

In [ ]:
# Observed difference
obs_diff = low_rates.mean() - high_rates.mean()
print(f'Observed difference: {obs_diff:.4f}')

# Permutation test
combined = np.concatenate([low_rates, high_rates])
n_low = len(low_rates)
n_reps = 10000
sim_diffs = []

for _ in range(n_reps):
    shuffled = np.random.permutation(combined)
    sim_diffs.append(shuffled[:n_low].mean() - shuffled[n_low:].mean())

p_value = np.mean(np.abs(sim_diffs) >= np.abs(obs_diff))
print(f'P-value (two-tailed): {p_value:.4f}')

In [ ]:
# Plot null distribution
plt.figure(figsize=(9, 4))
plt.hist(sim_diffs, bins=50, edgecolor='black', color='steelblue')
plt.axvline(obs_diff, color='red', linestyle='--',
            label=f'Observed = {obs_diff:.4f}')
plt.axvline(-obs_diff, color='red', linestyle='--')
plt.title('Permutation Distribution: Difference in Mean Food Insecurity Rate\n(Low − High Income)')
plt.xlabel('Simulated Difference in Mean Insecurity Rate')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Null distribution range: {min(sim_diffs):.4f} to {max(sim_diffs):.4f}')
print(f'Observed difference: {obs_diff:.4f} — far outside the null distribution')

**Q: How does the observed difference compare to the null distribution? What does the p-value suggest?**

**A:** The observed difference of 0.2047 falls completely outside the null distribution, which ranges from about −0.071 to +0.066. The p-value of 0.0000 means none of the 10,000 random shuffles produced a difference as large as the one observed. We reject the null hypothesis — the gap in food insecurity between low-income and high-income households is statistically significant and not due to chance.

---
## Step 6: Bootstrap Confidence Interval

We estimate the **median food insecurity rate** for the low-income group across survey weeks using bootstrapping.

**Why the CLT does not apply:** The CLT applies to means and proportions, but not to medians. With only 46 observations in the low-income group, we cannot assume the sampling distribution of the median is normal. Bootstrapping estimates the 95% confidence interval without that assumption.

In [ ]:
# Bootstrap the median insecurity rate for low-income group
n_boot = 10000
boot_medians = [
    np.median(np.random.choice(low_rates, size=len(low_rates), replace=True))
    for _ in range(n_boot)
]

ci_low = np.quantile(boot_medians, 0.025)
ci_high = np.quantile(boot_medians, 0.975)
boot_mean = np.mean(boot_medians)

print(f'Observed median insecurity rate (low-income): {np.median(low_rates):.4f}')
print(f'Bootstrap mean of medians: {boot_mean:.4f}')
print(f'95% Bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}]')

In [ ]:
# Plot bootstrap distribution
plt.figure(figsize=(8, 4))
plt.hist(boot_medians, bins=40, edgecolor='black', color='salmon')
plt.axvline(ci_low, color='red', linestyle='--',
            label=f'95% CI: [{ci_low:.4f}, {ci_high:.4f}]')
plt.axvline(ci_high, color='red', linestyle='--')
plt.axvline(boot_mean, color='black', linestyle='-',
            label=f'Mean of bootstrap medians: {boot_mean:.4f}')
plt.title('Bootstrap Distribution of Median Food Insecurity Rate\n(Low-Income Group)')
plt.xlabel('Bootstrap Median Insecurity Rate')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

**Q: What does the bootstrap confidence interval tell us?**

**A:** We are 95% confident that the true median weekly food insecurity rate for low-income households during 2020–2021 was between 18.8% and 25.7%. This entire interval is far above the high-income group mean of 2.0%, confirming that the gap is real and not driven by random variation. The bootstrap distribution is approximately symmetric, indicating the estimate is stable.

---
## Step 7: Only One Test (OOT) Framework

| Element | In this example |
|---|---|
| 1. Data | U.S. Household Pulse Survey — weekly food insecurity rates by income group, 2020–2021 |
| 2. Test statistic | Mean food insecurity rate (low-income) − mean food insecurity rate (high-income) |
| 3. Observed value | 0.2047 — low-income households had a 20.5 percentage point higher insecurity rate |
| 4. Null hypothesis | Income group has no effect on food insecurity rate; labels are effectively random |
| 5. How you simulate under the null | Pool all weekly rates, randomly shuffle low/high income labels, recompute the mean difference |
| 6. Reference distribution | Distribution of 10,000 simulated differences under the null, centered around 0, ranging from −0.071 to +0.066 |
| 7. P-value | 0.0000 — no simulated difference was as extreme as the observed one |

---
## Step 8: Reflection

**What did we find?**
Low-income households (under $35,000/year) reported food insecurity rates of roughly 22–28% across 2020–2021, compared to 2–4% for high-income households ($75,000+). The observed difference of 0.205 was far outside the null distribution (p = 0.0000), and the 95% bootstrap CI for the low-income median rate was [0.1883, 0.2570] — entirely above the high-income mean.

**How certain are we?**
Very certain. Both the permutation test and bootstrap CI point to the same conclusion: the gap is real, large, and statistically significant. The pattern was also consistent across all 22 survey weeks.

**Limitations:**
- Data is aggregated, not individual responses
- We cannot control for race, employment, or geography simultaneously
- Results are specific to the COVID-19 period and may not generalize

**What would we recommend?**
The data strongly supports expanding food assistance programs targeted at households below $35,000 during economic crises. Even with SNAP and stimulus payments in place during 2020–2021, the income gap in food insecurity remained large and persistent throughout the entire observation period.